# 01 - Data Extraction & Feature Engineering

Notebook này thực hiện ETL từ Data Warehouse:
1. Kết nối DB
2. Truy vấn dữ liệu với điều kiện `is_closed = false`
3. Xử lý missing values
4. Tạo feature `speed_ratio`, `speed_delta`
5. Ghép `ward_district_id`
6. Xuất `01_processed_features.parquet`

In [ ]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

In [ ]:
# ==== Config ====
DB_CONNECTION_STRING = os.getenv('DB_CONNECTION_STRING', '')
START_DATE = os.getenv('ETL_START_DATE', '2026-03-20')
END_DATE = os.getenv('ETL_END_DATE', '2026-04-08')
OUTPUT_PATH = '/workspace/ai-core/notebooks/01_processed_features.parquet'

# Bạn có thể đổi tên bảng nếu schema thực tế khác
FLOW_TABLE = 'fact_traffic_flow'
SEGMENT_TABLE = 'dim_segment'

assert DB_CONNECTION_STRING, 'Thiếu DB_CONNECTION_STRING trong environment'
print('Date range:', START_DATE, '->', END_DATE)

In [ ]:
engine = create_engine(DB_CONNECTION_STRING)

query = text(f'''
SELECT
    f.segment_key,
    f.timestamp,
    f.current_speed_kmh,
    f.traffic_index,
    f.delay_seconds,
    f.quality_flag,
    f.congestion_level,
    f.is_closed,
    s.free_flow_speed_kmh,
    s.default_lane_count,
    s.ward_district_id,
    s.tomtom_frc,
    s.is_one_way
FROM {FLOW_TABLE} f
LEFT JOIN {SEGMENT_TABLE} s
    ON s.segment_key = f.segment_key
WHERE f.timestamp >= :start_date
  AND f.timestamp < :end_date
  AND COALESCE(f.is_closed, false) = false
''')

with engine.connect() as conn:
    df_raw = pd.read_sql(query, conn, params={'start_date': START_DATE, 'end_date': END_DATE})

print('Extracted rows:', len(df_raw))
df_raw.head(3)

In [ ]:
# Parse timestamp and sort
df = df_raw.copy()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['segment_key', 'timestamp']).reset_index(drop=True)

# Ensure target range [0..5]
df['congestion_level'] = pd.to_numeric(df['congestion_level'], errors='coerce').fillna(0).clip(0, 5).astype(int)

# Missing handling strategy
num_fill_zero = ['traffic_index', 'delay_seconds', 'quality_flag']
num_ffill_bfill = ['current_speed_kmh', 'free_flow_speed_kmh', 'default_lane_count']
cat_ffill_bfill = ['ward_district_id', 'tomtom_frc', 'is_one_way']

for c in num_fill_zero:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)

for c in num_ffill_bfill:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
        df[c] = df.groupby('segment_key')[c].transform(lambda s: s.ffill().bfill())

for c in cat_ffill_bfill:
    if c in df.columns:
        df[c] = df.groupby('segment_key')[c].transform(lambda s: s.ffill().bfill())

# Fallback defaults
if 'free_flow_speed_kmh' in df.columns:
    df['free_flow_speed_kmh'] = df['free_flow_speed_kmh'].fillna(40.0)
if 'default_lane_count' in df.columns:
    df['default_lane_count'] = df['default_lane_count'].fillna(2.0)
if 'ward_district_id' in df.columns:
    df['ward_district_id'] = df['ward_district_id'].fillna(-1).astype(int)

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct.head(15)

In [ ]:
# Feature engineering
df['speed_ratio'] = (
    pd.to_numeric(df['current_speed_kmh'], errors='coerce') /
    pd.to_numeric(df['free_flow_speed_kmh'], errors='coerce').replace(0, np.nan)
)
df['speed_ratio'] = df['speed_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0)

df['speed_delta'] = (
    df.groupby('segment_key')['current_speed_kmh']
      .diff()
      .fillna(0.0)
)

# Ghép/chuẩn hóa ward_district_id (nếu missing sau join)
if 'ward_district_id' not in df.columns:
    df['ward_district_id'] = -1
df['ward_district_id'] = pd.to_numeric(df['ward_district_id'], errors='coerce').fillna(-1).astype(int)

# Cleanup columns
drop_cols = [c for c in ['is_closed'] if c in df.columns]
df = df.drop(columns=drop_cols)

df.head(5)

In [ ]:
# Final checks and export
required_cols = [
    'segment_key', 'timestamp', 'current_speed_kmh', 'traffic_index',
    'delay_seconds', 'quality_flag', 'free_flow_speed_kmh',
    'speed_ratio', 'speed_delta', 'ward_district_id', 'congestion_level'
]
missing_required = [c for c in required_cols if c not in df.columns]
assert not missing_required, f'Missing required columns: {missing_required}'

df.to_parquet(OUTPUT_PATH, index=False)
print('Saved:', OUTPUT_PATH)
print('Shape:', df.shape)
print('Class counts:\n', df['congestion_level'].value_counts().sort_index())